[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shreyashgol/phonetic-guided-speech-generation/blob/main/notebooks/16_final_report_generation.ipynb)

# 16 · Final report generation (Phase 1 + Phase 2)
**Goal:** turn the saved result files into everything the final IEEE report needs for Phase 2: macros with every number, tables, figures, the new LaTeX section and the conclusion. **Nothing is trained or tuned here**; every number comes from a file in `results/`, so the report is reproducible.

**Run order:** notebook 09 (Phase 1 figures and tables) → this notebook (Phase 2). No GPU needed.

**Design decisions (put these in the report)**
- **Numbers are never typed by hand.** They are read from `14_objective.csv`, `14_paired.csv`, `12_val_log.csv`, `13_copy_synthesis.csv` and the best checkpoint, and written to LaTeX macros.
- **Claims follow the statistics.** The results paragraph is generated from the sign and the interval of each paired difference ("closer than", "not distinguishable from", "practically equivalent to"). If a rerun changes the result, the wording changes with it.
- **The listening test is gated.** It goes into the report only if (a) you set `INCLUDE_LISTENING = True` and (b) the ratings pass automatic checks: at least 3 raters, inter-rater correlation at least 0.2, and no rater with block-filled scores. Otherwise the report says plainly that subjective evaluation is future work.
- **Overleaf-ready:** the notebook writes separate files and a zip, so your existing `main.tex` only needs a few one-line edits (listed at the end).

**Reads:** `results/14_*.csv`, `12_*`, `13_*`, `checkpoints/acoustic/best.pt`, `docs/architecture.png`. **Writes:** `reports/{results_macros_phase2.tex, phase2_section.tex, phase2_conclusion.tex, phase2_references.tex, PHASE2_PATCH.md, phase2_overleaf_bundle.zip}`, `reports/tables/phase2_*.tex`, `reports/figures/p2_*.png`.

In [ ]:
# ---- Cell 0: Setup (identical at the top of every notebook)
import os, sys, json, glob
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/phonetic-tts'
    IN_COLAB = True
except ImportError:                      # running somewhere else (local / tests)
    PROJECT_ROOT = os.environ.get('PROJECT_ROOT', os.getcwd())
    IN_COLAB = False
sys.path.insert(0, PROJECT_ROOT)
from src.utils.config import load_config
cfg = load_config(PROJECT_ROOT)
A = cfg['audio']; P = cfg['abs_paths']
RAW, PROCESSED, ALIGN, SPLITS, RESULTS = P['raw'], P['processed'], P['alignments'], P['splits'], P['results']
for d in (PROCESSED, ALIGN, SPLITS, RESULTS):
    os.makedirs(d, exist_ok=True)
print('Project root:', PROJECT_ROOT, '| Colab:', IN_COLAB)

Mounted at /content/drive
Project root: /content/drive/MyDrive/phonetic-tts | Colab: True


## Step 1: Helpers and folders

In [ ]:
import numpy as np, pandas as pd, shutil, zipfile, glob, textwrap
REP = os.path.join(PROJECT_ROOT, 'reports'); FIG = os.path.join(REP, 'figures'); TAB = os.path.join(REP, 'tables')
for d in (REP, FIG, TAB): os.makedirs(d, exist_ok=True)
FRAME_MS = 256 / 22050 * 1000
MISSING = []
def rd(name):
    p = os.path.join(RESULTS, name)
    if not os.path.exists(p): MISSING.append(name); print('MISSING result file:', name); return None
    return pd.read_csv(p)
def tex(df, name, caption, label, ff='%.3f'):
    d = df.copy(); d.columns = [str(c).replace('_', ' ') for c in d.columns]
    for c in d.columns:
        if not pd.api.types.is_numeric_dtype(d[c]): d[c] = d[c].astype(str).str.replace('_', ' ')
    t = d.to_latex(index=False, float_format=ff, caption=caption, label=label, position='!t')
    t = t.replace('\\begin{tabular}', '\\resizebox{\\columnwidth}{!}{\\begin{tabular}').replace('\\end{tabular}', '\\end{tabular}}')
    open(os.path.join(TAB, name), 'w').write(t); print('table ->', name)
def boot(x, B=1000, seed=0):
    r = np.random.RandomState(seed); x = np.asarray(x, float); bs = [x[r.randint(0, len(x), len(x))].mean() for _ in range(B)]
    return x.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)
print('report folder:', REP)


report folder: /content/drive/MyDrive/phonetic-tts/reports


## Step 2: Read the Phase 2 results and build the macros
Stops with a clear message if notebook 14 has not been run.

In [ ]:
# ---- Load Phase 2 results and turn them into LaTeX macros (every number comes from a file in results/)
T = rd('14_objective.csv'); P = rd('14_paired.csv'); VL = rd('12_val_log.csv'); CS = rd('13_copy_synthesis.csv'); PU = rd('14_objective_per_utt.csv')
assert T is not None and P is not None, 'Run notebook 14 first (14_objective.csv and 14_paired.csv are required).'
T = T.set_index('condition'); P = P.set_index('comparison')
M = {}                                                   # macro name -> LaTeX text
sg = lambda x, n=4: ('$-$' if x < 0 else '') + f'{abs(x):.{n}f}'
ci = lambda lo, hi: f'[{lo:.3f}, {hi:.3f}]'
for k, c in dict(GT='ground_truth', BL='phase1_bilstm', PM='phone_mean', GM='global_mean', OH='own_head').items():
    M['dtw' + k] = f'{T.loc[c, "dtw_l1"]:.3f}'; M['dtw' + k + 'ci'] = ci(T.loc[c, 'ci_lo'], T.loc[c, 'ci_hi'])
    M['len' + k] = f'{T.loc[c, "total_len_err_frames"]:.1f}'; M['len' + k + 'ms'] = f'{T.loc[c, "total_len_err_frames"] * FRAME_MS:.0f}'
pairs = dict(BP='phase1_bilstm - phone_mean', BG='phase1_bilstm - global_mean', BO='phase1_bilstm - own_head', BT='phase1_bilstm - ground_truth', PT='phone_mean - ground_truth')
for k, c in pairs.items():
    r = P.loc[c]; M['pair' + k] = sg(r.mean_diff); M['pair' + k + 'ci'] = '[' + sg(r.ci_lo) + ', ' + sg(r.ci_hi) + ']'; M['pair' + k + 'share'] = f'{100 * r.share_a_better:.1f}\\%'
gap = (P.loc[pairs['PT'], 'mean_diff'] - P.loc[pairs['BT'], 'mean_diff']) / P.loc[pairs['PT'], 'mean_diff'] * 100
rel = -P.loc[pairs['BP'], 'mean_diff'] / T.loc['phone_mean', 'dtw_l1'] * 100
M['gapClosed'] = f'{gap:.0f}\\%'; M['relEffect'] = f'{rel:.1f}\\%'
M['ptEvalN'] = str(int(PU.utt_id.nunique())) if PU is not None else '200'
M['csMelLone'] = f'{CS.mel_l1_vs_original.mean():.2f}' if CS is not None else '--'

# model facts: from the checkpoint if torch is available, else from the logs
n_par = step = best_val = None
try:
    import torch
    ck = torch.load(os.path.join(PROJECT_ROOT, 'checkpoints', 'acoustic', 'best.pt'), map_location='cpu', weights_only=False)
    n_par = sum(v.numel() for v in ck['model'].values()); step = int(ck['step']); best_val = float(ck['best'])
except Exception as e:
    print('checkpoint not read (', e, '); using logs')
if VL is not None:
    b = VL.loc[VL.val_mel_l1.idxmin()]; step = step or int(b.step); best_val = float(b.val_mel_l1); M['ptValDurMae'] = f'{b.val_dur_mae_frames:.2f}'
M['ptParams'] = f'{n_par / 1e6:.2f}' if n_par else '--'; M['ptStep'] = f'{step:,}'.replace(',', '{,}') if step else '--'; M['ptValMel'] = f'{best_val:.3f}' if best_val is not None else '--'

# data-driven wording: the claims follow the sign and the interval, not our hopes
def verdict(key, practical=0.003):
    r = P.loc[pairs[key]]
    if r.ci_hi < 0: return 'practically equivalent to' if abs(r.mean_diff) < practical else 'closer to the reference than'
    if r.ci_lo > 0: return 'farther from the reference than'
    return 'not distinguishable from'
d = T['dtw_l1']
M['ptFindings'] = (f'With {M["ptEvalN"]} test utterances, the mel distance was {d.phase1_bilstm:.3f} with Phase~1 BiLSTM durations, '
    f'{d.phone_mean:.3f} with phone-mean durations, {d.global_mean:.3f} with global-mean durations and {d.own_head:.3f} with the acoustic model\'s own duration head; '
    f'ground-truth durations gave {d.ground_truth:.3f}. The paired bootstrap shows that BiLSTM durations are {verdict("BP")} phone-mean durations '
    f'(difference {M["pairBP"]}, 95\\% CI {M["pairBPci"]}; closer on {M["pairBPshare"]} of utterances) and {verdict("BG")} global-mean durations '
    f'(difference {M["pairBG"]}, CI {M["pairBGci"]}). Against the model\'s own duration head the BiLSTM is {verdict("BO")} it '
    f'(difference {M["pairBO"]}, CI {M["pairBOci"]}; closer on {M["pairBOshare"]} of utterances). '
    f'The BiLSTM closes about {M["gapClosed"]} of the gap between the phone-mean baseline and ground-truth durations. '
    f'Total-length error falls from {M["lenPM"]} frames (phone mean) to {M["lenBL"]} frames (BiLSTM). '
    f'The absolute effect is small ({M["relEffect"]} of the phone-mean distance): most of the remaining distance is already present with ground-truth durations and is due to the acoustic model, not to timing.')
M['ptShort'] = (f'the Phase~1 BiLSTM durations were {verdict("BP")} phone-mean durations in mel distance ({M["pairBP"]}, 95\\% CI {M["pairBPci"]}), '
                f'while the effect size stayed small ({M["relEffect"]})')
pd.Series(M).to_frame('latex').head(12)


,latex
dtwGT,0.662
dtwGTci,"[0.656, 0.667]"
lenGT,0.0
lenGTms,0
dtwBL,0.665
dtwBLci,"[0.659, 0.670]"
lenBL,20.1
lenBLms,234
dtwPM,0.678
dtwPMci,"[0.673, 0.684]"


## Step 3: Listening test gate
Leave `INCLUDE_LISTENING = False` unless the group has re-rated every clip (one clip at a time) and merged the sheets into `results/listening/ratings.csv`. If you switch it on and the checks fail, the notebook refuses and tells you why.

In [ ]:
# ---- Listening test: included in the report ONLY if the ratings pass quality checks AND you switch the flag on
INCLUDE_LISTENING = False        # set to True only after the group has re-rated every clip, one clip at a time
rp = os.path.join(RESULTS, 'listening', 'ratings.csv'); kp = os.path.join(RESULTS, 'listening', 'PRIVATE_key.csv')
listening_ok = False; mos_tbl = None
if os.path.exists(rp) and os.path.exists(kp):
    rt = pd.read_csv(rp).dropna(subset=['score']); W = rt.pivot_table(index='code', columns='rater', values='score')
    n_r = W.shape[1]; cm = W.corr().to_numpy(); mean_corr = float(np.nanmean(cm[np.triu_indices(n_r, 1)])) if n_r > 1 else float('nan')
    rng = np.random.RandomState(0); ratios = []
    for r in W:
        s = W[r].to_numpy(); runs = 1 + (s[1:] != s[:-1]).sum()
        exp = np.mean([1 + (p[1:] != p[:-1]).sum() for p in (rng.permutation(s) for _ in range(500))]); ratios.append(runs / exp)
    checks = {'at least 3 raters': n_r >= 3, 'mean inter-rater correlation >= 0.2': mean_corr >= 0.2,
              'no rater has block-filled scores (runs/expected >= 0.6)': min(ratios) >= 0.6}
    print(f'raters {n_r} | mean inter-rater correlation {mean_corr:.2f} | runs/expected per rater {np.round(ratios, 2).tolist()}')
    for k, v in checks.items(): print(('PASS  ' if v else 'FAIL  ') + k)
    quality_ok = all(checks.values())
    if INCLUDE_LISTENING and not quality_ok: print('REFUSING to include the listening test: the quality checks failed. Re-collect the ratings.')
    listening_ok = INCLUDE_LISTENING and quality_ok
    if listening_ok:
        rt = rt.merge(pd.read_csv(kp), on='code'); pc = rt.groupby(['utt_id', 'cond']).score.mean().unstack()
        rows = []
        for c in pc.columns: m, lo, hi = boot(pc[c]); rows.append(dict(condition=c, mos=m, ci_lo=lo, ci_hi=hi))
        mos_tbl = pd.DataFrame(rows); dd = (pc['phase1_bilstm'] - pc['phone_mean']).to_numpy(); dm, dlo, dhi = boot(dd)
        M['listeningParagraph'] = (f'A blind listening test with {n_r} raters rated {len(pc)} test utterances in four conditions (natural speech through the vocoder, and synthesis with ground-truth, Phase~1 BiLSTM and phone-mean durations) on a 1--5 naturalness scale; scores were averaged over raters per clip and intervals are utterance-level bootstrap CIs (Table~\\ref{{tab:mos}}). '
            f'The paired difference BiLSTM minus phone mean was {sg(dm, 2)} (95\\% CI [{sg(dlo, 2)}, {sg(dhi, 2)}]). With {len(pc)} utterances per condition this test is small and should be read as indicative only.')
if not listening_ok:
    M['listeningParagraph'] = ('A blind listening test (10 test utterances $\\times$ 4 conditions) was prepared (notebook~14), but its ratings are not reported here because they did not meet our quality checks; '
        'subjective evaluation therefore remains future work, and the conclusions of this section rest on the objective measures above.')
print('listening test in report:', listening_ok)


raters 3 | mean inter-rater correlation -0.19 | runs/expected per rater [0.15, 0.14, 0.16]
PASS  at least 3 raters
FAIL  mean inter-rater correlation >= 0.2
FAIL  no rater has block-filled scores (runs/expected >= 0.6)
listening test in report: False


## Step 4: Tables and figures

In [ ]:
# ---- Tables
nice = {'ground_truth': 'ground truth (upper bound)', 'phase1_bilstm': 'Phase 1 BiLSTM', 'phone_mean': 'phone mean', 'global_mean': 'global mean', 'own_head': 'model own duration head'}
t = T.reset_index(); t['condition'] = t.condition.map(nice)
t = t.rename(columns={'dtw_l1': 'DTW mel L1', 'ci_lo': 'CI low', 'ci_hi': 'CI high', 'total_len_err_frames': 'length err (frames)'})[['condition', 'DTW mel L1', 'CI low', 'CI high', 'length err (frames)']]
tex(t, 'phase2_objective.tex', 'Duration-source ablation on test utterances: DTW log-mel L1 (lower is better) with 95\\% bootstrap CIs, and absolute error of total length.', 'tab:p2obj')
p = P.reset_index(); p['comparison'] = p.comparison.str.replace('phase1_bilstm', 'BiLSTM').str.replace('phone_mean', 'phone mean').str.replace('global_mean', 'global mean').str.replace('own_head', 'own head').str.replace('ground_truth', 'ground truth')
p = p.rename(columns={'mean_diff': 'mean diff', 'ci_lo': 'CI low', 'ci_hi': 'CI high', 'share_a_better': 'share first closer'})
tex(p, 'phase2_paired.tex', 'Paired differences in DTW mel L1 (negative: first condition is closer to the real mel).', 'tab:p2pair', ff='%.4f')
if mos_tbl is not None: tex(mos_tbl.round(2), 'phase2_mos.tex', 'Listening test: mean naturalness score (1--5) with 95\\% bootstrap CIs over utterances.', 'tab:mos')
# ---- Figures (copied from results/ so the report only uses generated files)
for src, dst in [('14_objective.png', 'p2_objective.png'), ('12_curves.png', 'p2_curves.png'), ('12_val_example.png', 'p2_val_example.png'), ('11_overfit_check.png', 'p2_overfit.png')]:
    s = os.path.join(RESULTS, src)
    if os.path.exists(s): shutil.copy(s, os.path.join(FIG, dst)); print('figure ->', dst)
    else: MISSING.append(src); print('MISSING figure:', src)
a = os.path.join(PROJECT_ROOT, 'docs', 'architecture.png')
if os.path.exists(a): shutil.copy(a, os.path.join(FIG, 'architecture.png')); print('figure -> architecture.png')
else: print('NOTE: put architecture.png in docs/ (or upload it to reports/figures/ yourself)')


table -> phase2_objective.tex
table -> phase2_paired.tex
figure -> p2_objective.png
figure -> p2_curves.png
figure -> p2_val_example.png
figure -> p2_overfit.png
NOTE: put architecture.png in docs/ (or upload it to reports/figures/ yourself)


## Step 5: LaTeX files (Phase 2 section, conclusion, references, macros)

In [ ]:
SECTION = r'''\section{Phase~2: Duration-Controlled Speech Synthesis}\label{sec:phase2}
Phase~2 asks whether better phoneme durations give better synthesised speech. We train one acoustic model on the Phase~1 alignments and change only the duration source at test time.

\subsection{System}
\figbox{architecture.png}{Overall system. Phase~1 (forced alignment and duration prediction) supplies the durations that the Phase~2 acoustic model and the pretrained vocoder turn into audio.}{p2arch}
The acoustic model follows FastSpeech~2 \cite{ren2021} without the pitch and energy predictors. Phoneme and stress embeddings with sinusoidal positions pass through four feed-forward Transformer (FFT) blocks (two-head self-attention and a one-dimensional convolutional feed-forward layer). A length regulator repeats each token vector by its duration, and four further FFT blocks produce an 80-bin mel spectrogram. The model has \ptParams{}~M parameters (hidden size 256). A small duration head (two convolutional layers predicting $\log(d{+}1)$) is trained jointly. Training uses teacher forcing: the length regulator receives the forced-alignment durations from Phase~1. Silence is kept as a token, so the durations of an utterance sum exactly to its mel length. The mel recipe (22{,}050~Hz, FFT 1024, hop 256, 80 bins, natural log) is the one the pretrained vocoder expects, and durations are in mel frames (11.6~ms), the same unit as the Phase~1 targets.

\subsection{Training and vocoder}
We use AdamW (peak learning rate $5\!\times\!10^{-4}$, 1{,}000 warm-up steps, inverse-square-root decay), batch size 16 grouped by length, mixed precision and gradient clipping at 1.0. The loss is the masked $L_1$ distance on the normalised mel plus the masked mean squared error of the log-duration. The best checkpoint (step \ptStep{}, validation mel $L_1$ \ptValMel{} in normalised units) was selected on the validation split only. Splits are the frozen Phase~1 splits. Audio is rendered by a pretrained HiFi-GAN \cite{kong2020} (SpeechBrain \cite{ravanelli2021}, trained on LJSpeech), which we did not train. Re-synthesising real clips through our mel recipe and this vocoder gave a mean log-mel $L_1$ of \csMelLone{} against the originals, and the output sounded clean on inspection.

\subsection{Duration-source experiment}
The same trained model synthesised each test utterance with five duration sources: (i)~ground truth (upper bound), (ii)~the Phase~1 BiLSTM, (iii)~the phone-mean baseline, (iv)~the global mean, and (v)~the acoustic model's own duration head. Phase~1 predicts speech phones only, so silence tokens keep their ground-truth length in every condition and only speech-phone durations differ. Because different durations give different lengths, we compare the synthesised mel $\hat M$ with the real mel $M$ after dynamic time warping \cite{sakoe1978}:
\begin{equation}
D(\hat M, M)=\frac{1}{|\pi^\star|\,F}\sum_{(i,j)\in\pi^\star}\lVert \hat m_i-m_j\rVert_1,\quad F=80,
\end{equation}
where $\pi^\star$ is the minimum-cost warping path. We also report the absolute error of total length. Intervals are 95\% bootstrap CIs over \ptEvalN{} test utterances, and conditions are compared by paired bootstrap differences on the same utterances.

\subsection{Results}
\ptFindings{}

\safeinput{tables/phase2_objective.tex}
\safeinput{tables/phase2_paired.tex}
\figbox{p2_objective.png}{Duration-source ablation: DTW mel distance (left) and absolute total-length error (right), with 95\% bootstrap intervals.}{p2obj}
\figbox{p2_curves.png}{Training and validation curves of the acoustic model.}{p2curves}

\subsection{Subjective evaluation}
\listeningParagraph{}
\IfFileExists{tables/phase2_mos.tex}{\input{tables/phase2_mos.tex}}{}

\subsection{Limitations of Phase~2}
\begin{itemize}
\item The DTW mel distance is a proxy for quality: it removes timing differences, so it measures spectral similarity given the durations, not naturalness.
\item The acoustic model is small and has no pitch or energy predictors; its own mel error dominates the remaining distance, which limits how much better durations can help.
\item The vocoder is pretrained, the data are one speaker of read English, and the experiment uses \ptEvalN{} test utterances.
\item The text demonstration uses the model's own duration head; driving it with the Phase~1 BiLSTM on free text requires exposing the Phase~1 feature builder as a function (future work).
\end{itemize}
'''

In [ ]:
CONCLUSION = r'''\section{Conclusion}
Phase~1 delivers validated duration labels (\nKept{} utterances), a frozen split and a documented comparison of duration predictors (best: \bestModel, MAE \maeBest{} frames).
Phase~2 feeds phonemes and durations through a length regulator into a small FastSpeech~2-style mel decoder and a pretrained HiFi-GAN vocoder. In a controlled duration-source experiment on the same trained model, \ptShort{}. Future work: pitch and energy predictors, a larger and fully blind listening study, and running the Phase~1 predictor directly on free text.
'''

In [ ]:
REFS = r'''\bibitem{ravanelli2021} M.~Ravanelli \emph{et al.}, ``SpeechBrain: A general-purpose speech toolkit,'' \emph{arXiv:2106.04624}, 2021.
\bibitem{sakoe1978} H.~Sakoe and S.~Chiba, ``Dynamic programming algorithm optimization for spoken word recognition,'' \emph{IEEE Trans. Acoust., Speech, Signal Process.}, vol.~26, no.~1, pp.\ 43--49, 1978.
'''

In [ ]:
# ---- Write the LaTeX files (Overleaf-ready) and the macro file
MAC = os.path.join(REP, 'results_macros_phase2.tex')
lines = ['% Auto-generated by notebook 16 from results/*.csv. Do not edit by hand; rerun the notebook.']
for k, v in M.items(): lines.append('\\providecommand{\\%s}{}\\renewcommand{\\%s}{%s}' % (k, k, v))
open(MAC, 'w').write(chr(10).join(lines) + chr(10))
open(os.path.join(REP, 'phase2_section.tex'), 'w').write(SECTION)
open(os.path.join(REP, 'phase2_conclusion.tex'), 'w').write(CONCLUSION)
open(os.path.join(REP, 'phase2_references.tex'), 'w').write(REFS)
print('wrote macros:', len(M), '| section | conclusion | references')
PATCH = r'''# Edits to make in your Overleaf main.tex (once)
1. Directly after the line   \input{results_macros.tex}   add:   \input{results_macros_phase2.tex}
2. Directly before  \section{Conclusion and Phase~2 Outlook}  add:   \input{phase2_section.tex}
3. Replace the whole "Conclusion and Phase~2 Outlook" section (heading and paragraph) with:   \input{phase2_conclusion.tex}
4. In the bibliography, before \end{thebibliography}, paste the two lines from phase2_references.tex
5. In Reproducibility: change "notebooks 00 -> 09" to "notebooks 00 -> 16", and "every figure and table of this report is regenerated by notebook 09" to "by notebooks 09 and 16".
6. In Discussion and Limitations: delete the bullet "No audio yet ..." (Phase 2 now produces audio).
7. Abstract and Introduction: add one sentence that Phase 2 (acoustic model + pretrained vocoder + duration-source experiment) is included.
8. Contributions table: add the Phase 2 rows (see CONTRIBUTIONS.md) for the notebooks 10-16.
9. Upload to Overleaf: reports/figures/*, reports/tables/phase2_*.tex, the four new .tex files, then recompile.
'''
open(os.path.join(REP, 'PHASE2_PATCH.md'), 'w').write(PATCH)
print(PATCH)


wrote macros: 46 | section | conclusion | references
# Edits to make in your Overleaf main.tex (once)
1. Directly after the line   \input{results_macros.tex}   add:   \input{results_macros_phase2.tex}
2. Directly before  \section{Conclusion and Phase~2 Outlook}  add:   \input{phase2_section.tex}
3. Replace the whole "Conclusion and Phase~2 Outlook" section (heading and paragraph) with:   \input{phase2_conclusion.tex}
4. In the bibliography, before \end{thebibliography}, paste the two lines from phase2_references.tex
5. In Reproducibility: change "notebooks 00 -> 09" to "notebooks 00 -> 16", and "every figure and table of this report is regenerated by notebook 09" to "by notebooks 09 and 16".
6. In Discussion and Limitations: delete the bullet "No audio yet ..." (Phase 2 now produces audio).
7. Abstract and Introduction: add one sentence that Phase 2 (acoustic model + pretrained vocoder + duration-source experiment) is included.
8. Contributions table: add the Phase 2 rows (see CONTRIBU

## Step 6: Bundle for Overleaf and final check

In [ ]:
# ---- Bundle everything for Overleaf and run a final completeness check
zp = os.path.join(REP, 'phase2_overleaf_bundle.zip')
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ['results_macros_phase2.tex', 'phase2_section.tex', 'phase2_conclusion.tex', 'phase2_references.tex', 'PHASE2_PATCH.md']: z.write(os.path.join(REP, f), f)
    for f in glob.glob(os.path.join(TAB, 'phase2_*.tex')): z.write(f, 'tables/' + os.path.basename(f))
    for f in glob.glob(os.path.join(FIG, 'p2_*.png')) + glob.glob(os.path.join(FIG, 'architecture.png')): z.write(f, 'figures/' + os.path.basename(f))
print('bundle ->', zp); print(*zipfile.ZipFile(zp).namelist(), sep=chr(10))
need = ['figures/architecture.png', 'figures/p2_objective.png', 'figures/p2_curves.png', 'tables/phase2_objective.tex', 'tables/phase2_paired.tex']
miss = [n for n in need if not os.path.exists(os.path.join(REP, n))]
print(chr(10) + ('ALL REQUIRED FILES PRESENT' if not miss else 'MISSING: ' + ', '.join(miss)), '| listening test included:', listening_ok)


bundle -> /content/drive/MyDrive/phonetic-tts/reports/phase2_overleaf_bundle.zip
results_macros_phase2.tex
phase2_section.tex
phase2_conclusion.tex
phase2_references.tex
PHASE2_PATCH.md
tables/phase2_objective.tex
tables/phase2_paired.tex
figures/p2_objective.png
figures/p2_curves.png
figures/p2_val_example.png
figures/p2_overfit.png
figures/architecture.png

ALL REQUIRED FILES PRESENT | listening test included: False


## What to do next
1. Download `reports/phase2_overleaf_bundle.zip` from Drive and upload its contents into your Overleaf project (keep the `figures/` and `tables/` folders).
2. Make the edits listed in `PHASE2_PATCH.md` in `main.tex` (about five minutes).
3. Recompile. Any red `[missing ...]` box means a file is missing: rerun the notebook that writes it (09 for Phase 1, 14 for Phase 2).
4. Read the generated results paragraph once and make sure you can explain every sentence in the viva.

**Viva reminders:** the effect of better durations is *small but measurable*; the acoustic model's own error dominates; the model's own duration head performs on par with our BiLSTM; the vocoder is pretrained; the listening test is small or not reported, and why.